In [0]:
'''
from pyspark.sql.functions import col, year, lit,concat, expr
from pyspark.sql.types import DateType,IntegerType,StringType,StructType,StructField
dbutils.widgets.dropdown("environment", "test",["test","prod"])
env =dbutils.widgets.get("environment")

base_path =f"s3://fraud-transection-detection/{env}"
bronze_path =f"{base_path}/bronze"
silver_path =f"{base_path}/silver"


def micro_processing(batch_df,batch_id):
    batch_df.cache()

    good_data =batch_df.filter(
        (col("rescued_data").isNotNull()) &
        (col("amount").isNull()) & (col("amount")<= 0) & 
        (col("tr_date").isNull())
    )

    bad_data =batch_df.filter(
        (col("rescued_data").isNotNull())|
        (col("amount").isNull())|
        (col("tr_date").isNull())
    )

    good_data.write.format("delta").mode("append").saveAsTable(f"{silver_path}/silver_data")
    bad_data.write.format("delta").mode("append").saveAsTable(f"{silver_path}/silver_quarantine")

    batch_df.unpersist()

silver_schema =StructType([
    StructField("amount",IntegerType()),
    StructField("tr_date",DateType()),
    StructField("tr_id",StringType()),
    StructField("user_id",StringType()),
    StructField("merchant_id",StringType()),
    StructField("rescued_data",StringType())

])


#raw_data = spark.readStream.table(f"fraud_{env}.bronze").schema(silver_schema)
#When you call spark.readStream.table("name"), Spark immediately looks at the Hive Metastore to find the existing schema. If you try to #attach a .schema() after that, Spark gets confused because it thinks you are trying to "re-define" an already existing table structure.


raw_data = (spark.readStream
            .format("delta")
            .schema(silver_schema) # 1. Schema first
            .load(f"{bronze_path}")   # 2. Path is managed so overlap error 
)



raw_data.writeStream\
    .option("checkpointLocation",f"{silver_path}/checkpoint/data_splitter")\
    .foreachBatch(micro_processing)\
    .queryName("raw_data_splitter")\
    .start()

'''

In [0]:

from pyspark.sql.functions import col, year, lit,concat, expr,coalesce,current_date
from pyspark.sql.types import DateType,IntegerType,StringType,StructType,StructField
dbutils.widgets.dropdown("environment", "test",["test","prod"])
env =dbutils.widgets.get("environment")

base_path =f"s3://fraud-transection-detection/{env}"
bronze_table =f"{base_path}/bronze"
silver_path =f"{base_path}/silver"

silver_schema =StructType([
    StructField("amount",IntegerType()),
    StructField("tr_date",DateType()),
    StructField("tr_id",StringType()),
    StructField("user_id",StringType()),
    StructField("merchant_id",StringType())

])

#bronze_df = spark.readStream.table(f"fraud_{env}.bronze") #.schema(silver_schema).writeStream(format="delta",path=silver_path)
bronze_df = spark.readStream.table(f"fraud_{env}.bronze")


def micro_processing(batch_df,batch_id):
 #batch_df.cache()
 cleaned_df =batch_df.select(
    expr("try_cast(amount as int)").alias("amount"),
    expr("try_cast(tr_date as date)").alias("tr_date"),
    expr("try_cast(tr_id as string)").alias("tr_id"),  #col("tr_id").cast(StringType()),
    col("user_id").cast(StringType()),
    col("merchant_id").cast(StringType()),
    col("rescued_data")
    ).withColumn("tr_id",concat(year(col("tr_date")),lit('_'),col('tr_id')))\
    .filter((col("rescued_data").isNull()) & (col("amount")>0) & (col("amount").isNotNull()) & (col("tr_date").isNotNull())& (col("tr_date")<= current_date()))\
    .dropDuplicates(['tr_id','user_id'])


 bad_data =batch_df.select("*")\
    .filter((col("rescued_data").isNotNull()) | (col("amount").isNull()) | (col("tr_date").isNull()))


 cleaned_df.write.format("delta").mode("append")\
    .partitionBy("tr_date")\
    .save(f"{silver_path}/silver_data")

 bad_data.write.format("delta").mode("append").save(f"{silver_path}/silver_quarantine")
 #batch_df.unpersist()

bronze_df.writeStream\
    .format("delta")\
    .trigger(availableNow=True)\
    .option("checkpointLocation",f"{silver_path}/checkpoint/data_filteration")\
    .foreachBatch(micro_processing)\
    .queryName("data_filteration")\
    .start()
    


In [0]:


checkpoint_path = f"{silver_path}/checkpoint/silver_display_test"
checkpoint_path1 = f"{silver_path}/checkpoint/silver_display_test1"
dbutils.fs.rm(checkpoint_path, recurse=True)  #-- for deleting the previous checkpoints of earlier select or modification in select()

display(cleaned_df, mode="append", checkpointLocation=checkpoint_path)
display(bad_data, mode="append", checkpointLocation=checkpoint_path1)